In [1]:
import cv2
import numpy as np
import os
import csv
import json


IMG_FOLDER = r"C:\Users\marc\Downloads\dataset-master\dataset-master\JPEGImages"
GT_FOLDER  = r"C:\Users\marc\Downloads\dataset-master\dataset-master\GroundTruth"  
OUTPUT_FOLDER = r"C:\Users\marc\Downloads\dataset-master\dataset-master\detection_results"  
REPORT_PATH_CSV = r"C:\Users\marc\Downloads\dataset-master\dataset-master\evaluation_report.csv"
REPORT_PATH_JSON = r"C:\Users\marc\Downloads\dataset-master\dataset-master\evaluation_report.json"


IOU_THRESHOLD = 0.5


def load_image(filepath):
    return cv2.imread(filepath)

def ensure_image_size_matches(img, mask):
    
    if img.shape[:2] != mask.shape[:2]:
        mask = cv2.resize(mask, (img.shape<a href="" class="citation-link" target="_blank" style="vertical-align: super; font-size: 0.8em; margin-left: 3px;">[1]</a>, img.shape<a href="" class="citation-link" target="_blank" style="vertical-align: super; font-size: 0.8em; margin-left: 3px;">[0]</a>), interpolation=cv2.INTER_NEAREST)
    return mask

def extract_predicted_mask_from_contours(img_shape, markers, labels_to_skip={0,1}):
    """
    Given a Watershed markers image (same size as input),
    generate a binary mask where each non-background label (>1) is set.
    """
    pred_mask = np.zeros(img_shape[:2], dtype=np.uint8)
    
    unique = np.unique(markers)
    for lbl in unique:
        if lbl <= 1:
            continue
        pred_mask[markers == lbl] = 255
    return pred_mask

def iou(mask_pred, mask_gt):
    
    if mask_pred.dtype != np.uint8:
        mask_pred = mask_pred.astype(np.uint8)
    if mask_gt.dtype != np.uint8:
        mask_gt = mask_gt.astype(np.uint8)
    
    pred_bin = (mask_pred > 0).astype(np.uint8)
    gt_bin   = (mask_gt > 0).astype(np.uint8)
    intersection = np.logical_and(pred_bin, gt_bin).sum()
    union = np.logical_or(pred_bin, gt_bin).sum()
    if union == 0:
        return 0.0
    return float(intersection) / float(union)


def evaluate_image(img_name, img_path, gt_mask_path, pred_mask_path):
   
    gt_mask = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE)
    if gt_mask is None:
        raise FileNotFoundError(f"Ground truth mask not found: {gt_mask_path}")
    img = cv2.imread(img_path)
    gt_mask = ensure_image_size_matches(img, gt_mask)

    
    pred_mask = cv2.imread(pred_mask_path, cv2.IMREAD_GRAYSCALE)
    if pred_mask is None:
        
        pred_mask = np.zeros(gt_mask.shape, dtype=np.uint8)
    pred_mask = ensure_image_size_matches(img, pred_mask)

    
    num_gt, gt_labels, stats, centroids = cv2.connectedComponentsWithStats(gt_mask, connectivity=8)
   
    num_pred, pred_labels, pred_stats, pred_centroids = cv2.connectedComponentsWithStats(pred_mask, connectivity=8)

    
    gt_components = []
    for i in range(1, num_gt):  # skip background
        comp_mask = (gt_labels == i).astype(np.uint8) * 255
        gt_components.append(comp_mask)

    pred_components = []
    for j in range(1, num_pred):
        comp_mask = (pred_labels == j).astype(np.uint8) * 255
        pred_components.append(comp_mask)

   
    matched_gt = set()
    tp = 0
    total_pred = max(0, num_pred - 1)
    per_pred_iou = []

    for j, p_mask in enumerate(pred_components, start=1):
        best_iou = 0.0
        best_i = -1
        for i, g_mask in enumerate(gt_components, start=1):
            inter = cv2.bitwise_and(p_mask, g_mask)
            i_area = int(cv2.countNonZero(p_mask))
            if i_area == 0:
                i_area = 1
            iou_val = float(cv2.countNonZero(inter)) / float(cv2.countNonZero(cv2.bitwise_or(p_mask, g_mask)))
            if iou_val > best_iou:
                best_iou = iou_val
                best_i = i
        per_pred_iou.append(best_iou)
        if best_iou >= IOU_THRESHOLD:
            tp += 1
            if best_i > 0:
                matched_gt.add(best_i)

    fn = len(gt_components) - len(matched_gt)
    fp = total_pred - tp
    
    if len(gt_components) == 0:
        fn = 0
    if total_pred == 0:
        tp = 0
        fp = 0

    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    
    return {
        "image": img_name,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "num_gt": int(len(gt_components)),
        "num_pred": int(total_pred),
        "iou_mean": float(np.mean(per_pred_iou)) if per_pred_iou else 0.0
    }


def run_evaluation(img_folder, gt_folder, pred_folder):
    
    image_names = [f for f in os.listdir(img_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    results = []
    for name in image_names:
        img_path = os.path.join(img_folder, name)
        gt_mask_path = os.path.join(gt_folder, os.path.splitext(name)<a href="" class="citation-link" target="_blank" style="vertical-align: super; font-size: 0.8em; margin-left: 3px;">[0]</a> + ".png")
        
        pred_mask_path = os.path.join(pred_folder, os.path.splitext(name)<a href="" class="citation-link" target="_blank" style="vertical-align: super; font-size: 0.8em; margin-left: 3px;">[0]</a> + "_pred_mask.png")
        
        if not os.path.exists(pred_mask_path):
           
            print(f"Warning: predicted mask not found for {name}, skipping in evaluation.")
            continue

        res = evaluate_image(name, img_path, gt_mask_path, pred_mask_path)
        results.append(res)

    return results

def save_csv(results, path):
    if not results:
        return
    fieldnames = ["image","tp","fp","fn","precision","recall","f1","num_gt","num_pred","iou_mean"]
    with open(path, mode="w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow(r)

def save_json(results, path):
    with open(path, "w") as f:
        json.dump(results, f, indent=2)

def summarize(results):
    if not results:
        return {}
    total_tp = sum(r["tp"] for r in results)
    total_fp = sum(r["fp"] for r in results)
    total_fn = sum(r["fn"] for r in results)
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    iou_mean = np.mean([r["iou_mean"] for r in results]) if results else 0.0

    return {
        "total_images": len(results),
        "total_tp": int(total_tp),
        "total_fp": int(total_fp),
        "total_fn": int(total_fn),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "overall_iou_mean": float(iou_mean)
    }

if __name__ == "__main__":
    
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    
    results = run_evaluation(IMG_FOLDER, GT_FOLDER, OUTPUT_FOLDER)

    save_csv(results, REPORT_PATH_CSV)
    save_json(results, REPORT_PATH_JSON)

   
    summary = summarize(results)
    print("Evaluation Summary:")
    for k, v in summary.items():
        print(f"  {k}: {v}")



SyntaxError: invalid syntax. Perhaps you forgot a comma? (1190694558.py, line 24)